## Imports

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

import mesmer
import model

import xarray as xr

import numpy as np


## Load Data

In [2]:
is_local_data = False
Month_idx = 4
safe = True
start = None
end = None
max_depth = 3

In [3]:
raw_mrsol_for_mean = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=1)

raw_mrsol_for_var = model.loading.load_data_set(var = "mrsol", local = is_local_data, run_idx=2)

In [4]:
raw_mrsol_empirical_maximas_mean = raw_mrsol_for_mean.max("time")
raw_mrsol_empirical_maximas_var = raw_mrsol_for_var.max("time")

raw_mrsol_empirical_maximas = xr.concat([raw_mrsol_empirical_maximas_mean,raw_mrsol_empirical_maximas_var],dim="sets").max("sets")

approx_maximas = raw_mrsol_empirical_maximas.copy(deep=True)
approx_maximas["mrsol"] = (raw_mrsol_empirical_maximas["mrsol"] * 1.1).clip(min=1e-5)


In [ ]:
tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=1)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=1)

raw_input_for_mean = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

tas_ds = model.loading.load_data_set(var = "tas", local = is_local_data, run_idx=2)
pr_ds = model.loading.load_data_set(var = "pr", local = is_local_data, run_idx=2)

raw_input_for_var = xr.merge([tas_ds,pr_ds]).drop_vars("file_qf")

### some functions to shape the data


In [6]:
def shape_target(ds, maximas):
    ds = ds.clip(min=0)
    ds = ds/maximas
    ds = ds.sel(time=slice(start, end)).resample(time="ME").mean()
    ds = ds.sel(time= ds.time.dt.month == Month_idx).isel(depth=slice(0, max_depth))
    return ds

In [7]:
def mask_stack_target(ds):
    masked_ds, chunk_mask, detail_mask= model.mask.mask_nonpositiv_height_chunks(ds.drop_vars("depth_bnds"))
    ds = mesmer.grid.stack_lat_lon(masked_ds)
    ds = ds.transpose("gridcell","time", "depth")
    return ds,chunk_mask, detail_mask

In [8]:
def shape_input(ds, chunk_mask):
    ds = ds.sel(time=slice(start, end)).resample(time="ME").mean()
    ds = ds.sel(time= ds.time.dt.month == Month_idx)
    ds = model.mask.mask_mask(ds,chunk_mask)
    ds = mesmer.grid.stack_lat_lon(ds)
    return ds

    

### Linear Regression of the mean

In [9]:
mean_target = shape_target(raw_mrsol_for_mean, approx_maximas)
mean_target, chunk_mask, detail_mask = mask_stack_target(mean_target)
mean_target_da = mean_target.mrsol

In [10]:
mean_predictors = shape_input(raw_input_for_mean,chunk_mask)

In [11]:
LinReg_mean = model.stats._parallel_linear_regression.ParLinearRegression()

In [12]:
LinReg_mean.fit(predictors=mean_predictors, target=mean_target_da,location_dim="gridcell", regr_dim="time")

### Compute Residuals for the variance

In [13]:
var_target = shape_target(raw_mrsol_for_var, approx_maximas)
var_target, chunk_mask_var, detail_mask_var = mask_stack_target(var_target)


In [14]:
var_predictors = shape_input(raw_input_for_var,chunk_mask)

In [15]:
raw_residuals = LinReg_mean.residuals(var_predictors, var_target)

In [16]:
variance_estimator_set = {}

variance_estimator_set["quad_res"] = (raw_residuals.residuals)**2

variance_estimator_set["log_quad_res"] = np.log((raw_residuals.residuals)**2)


### Linear Regression of the Variance

In [17]:
Regr_set_var = {}

In [18]:
for var_key, var_est in variance_estimator_set.items():
    Regr_set_var[var_key] = model.stats._parallel_linear_regression.ParLinearRegression()
    Regr_set_var[var_key].fit(predictors=var_predictors, target=var_est,location_dim="gridcell", regr_dim="time")

### Export Prameters

In [19]:
if safe:
    for var_key, var_est in variance_estimator_set.items():
        model.save.save_params(LinReg_mean.params,Regr_set_var[var_key].params,maximas=approx_maximas,chunk_mask=chunk_mask,detail_mask=detail_mask,var = "mrsol",scen= "historical", folder = f"MPI-ESM1-2-LR/variations/variance_transform/{var_key}_rel_transform/local{is_local_data}/month{Month_idx}", name=f"start={start},end={end}.max_depth{max_depth}")

/home/pjunghans/BachelorArbeit/model/save.py:41: UserWarning: Unlimited dimension(s) {'time'} declared in 'dataset.encoding', but not part of current dataset dimensions. Consider removing {'time'} from 'dataset.encoding'.
  maximas.to_netcdf(out_max)


In [20]:
#model.loading.load_params(var = "mrsol",scen= "historical", folder = "MPI-ESM1-2-LR/variations/Transformation_Distribution", name=f"dist_beta_transform_rel_month_idx= {month_idx}")